In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import time

In [ ]:
from MMSNet import MMSDataset

dataset = MMSDataset(1000)
dataloader = DataLoader(dataset , batch_size=1, shuffle=False, num_workers=1)

In [ ]:
from pynq import get_rails, DataRecorder

rails = get_rails()
recorder = DataRecorder(rails['12V'].power)

# Test CPU

In [ ]:
from MMSNet import LogisticNet, ReducedNet, BaselineNet

model1 = LogisticNet()
model2 = ReducedNet()
model3 = BaselineNet()

model1.load_state_dict(torch.load('PreTrained_Weights/LogisticNet_84.pth'))
model2.load_state_dict(torch.load('PreTrained_Weights/ReducedNet_42.pth'))
model3.load_state_dict(torch.load('PreTrained_Weights/BaselineNet_336.pth'))

In [ ]:
recorder.record(0.2)
time.sleep(3)

In [ ]:
cpu_output_dict = {}
for model in [model1, model2, model3]:
    output_dict_key = f"{model.__class__.__name__}"
    model.eval()
    recorder.mark()
    with torch.no_grad():
        inference_time = 0
        cpu_output_dict[output_dict_key] = []
        for data in dataloader:
            # Run inference for each image individually
            start_time = time.time()
            pred = model(data)
            inference_time += time.time() - start_time
            cpu_output_dict[output_dict_key].append(pred.numpy().flatten())

        # Convert list to numpy array
        cpu_output_dict[output_dict_key] = np.array(cpu_output_dict[output_dict_key])

        # calculate average inference time and FPS
        num_images = len(cpu_output_dict[output_dict_key])
        avg_inference_time = inference_time / num_images
        fps = num_images / inference_time

        print(f"The NN under test is {output_dict_key}.")
        print(f"The total inference time was {inference_time:.6f} seconds for {num_images}.")
        print("Average inference time per image: {:.6f} seconds".format(avg_inference_time))
        print("FPS: {:.2f}".format(fps))
    
recorder.mark()

# Test DPU

In [ ]:
# VitisAI does not support the following operations:
# Pool3D and Conv3D can't be converted to XIR.

# Test FINN

In [ ]:
# FINN does not support the following operations:
# Con3D and Pool3D

# Test HLS

In [ ]:
from pynq import Overlay
from pynq import allocate

ol = Overlay(f"hls_bitstream/LogisticNet.bit")
py_buffer = allocate(shape=(1, 1, 32,16,32), dtype=np.float32)

def load_ip_input(UUT_ip, input_data):
    input_data_np = input_data.numpy()
    UUT_ip.write(0x10, py_buffer.physical_address)
    py_buffer[0] = input_data_np
    py_buffer.flush()
    

def get_ip_output(UUT_ip):
    output_1 = np.frombuffer(UUT_ip.read(0x20).to_bytes(4, 'little'), dtype=np.float32)
    output_2 = np.frombuffer(UUT_ip.read(0x24).to_bytes(4, 'little'), dtype=np.float32)
    output_3 = np.frombuffer(UUT_ip.read(0x28).to_bytes(4, 'little'), dtype=np.float32)
    output_4 = np.frombuffer(UUT_ip.read(0x2c).to_bytes(4, 'little'), dtype=np.float32)
    return np.concatenate((output_1, output_2, output_3, output_4), dtype=np.float32)

def start_ip(UUT_ip):
    UUT_ip.write(0x00, 0x01)
    while UUT_ip.read(0x00) & 0x4 == 0:
        continue

def run_ip(UUT_ip, input_data):
    load_ip_input(input_data)
    start_ip()
    return get_ip_output()

In [ ]:
hls_output_dict = {}
for model in [model1, model2]:
    output_dict_key = f"{model.__class__.__name__}"
    ol = Overlay(f"hls_bitstream/{output_dict_key}.bit")
    UUT_ip = ol.entry_0
    recorder.mark()
    inference_time = 0
    hls_output_dict[output_dict_key] = []
    for data in dataloader:
        load_ip_input(UUT_ip, data)
        start_time = time.time()
        start_ip(UUT_ip)
        inference_time += time.time() - start_time
        out_data = get_ip_output(UUT_ip)
        hls_output_dict[output_dict_key].append(out_data)

    # Convert list to numpy array
    hls_output_dict[output_dict_key] = np.array(hls_output_dict[output_dict_key])

    # calculate average inference time and FPS
    num_images = len(hls_output_dict[output_dict_key])
    avg_inference_time = inference_time / num_images
    fps = num_images / inference_time

    print(f"The NN under test is {output_dict_key}.")
    print(f"The total inference time was {inference_time:.6f} seconds for {num_images}.")
    print("Average inference time per image: {:.6f} seconds".format(avg_inference_time))
    print("FPS: {:.2f}".format(fps))

recorder.mark()

In [ ]:
output_dict_key = f"{model3.__class__.__name__}"
ol = Overlay(f"hls_bitstream/{output_dict_key}.bit")
UUT_ip = ol.entry_0

weight_cv2 = model3.state_dict()["cv2.weight"]
weight_fc1 = model3.state_dict()["fc1.weight"]
tensor_cv2_weight_buffer = allocate(shape=weight_cv2.shape, dtype=np.float32)
tensor_fc1_weight_buffer = allocate(shape=weight_fc1.shape, dtype=np.float32)
tensor_cv2_weight_buffer[:] = weight_cv2.numpy()
tensor_fc1_weight_buffer[:] = weight_fc1.numpy()
UUT_ip.write(0x30, tensor_cv2_weight_buffer.physical_address)
UUT_ip.write(0x3c, tensor_fc1_weight_buffer.physical_address)

recorder.mark()
inference_time = 0
hls_output_dict[output_dict_key] = []
for data in dataloader:
    load_ip_input(UUT_ip, data)
    start_time = time.time()
    start_ip(UUT_ip)
    inference_time += time.time() - start_time
    out_data = get_ip_output(UUT_ip)
    hls_output_dict[output_dict_key].append(out_data)

# Convert list to numpy array
hls_output_dict[output_dict_key] = np.array(hls_output_dict[output_dict_key])

# calculate average inference time and FPS
num_images = len(hls_output_dict[output_dict_key])
avg_inference_time = inference_time / num_images
fps = num_images / inference_time

print(f"The NN under test is {output_dict_key}.")
print(f"The total inference time was {inference_time:.6f} seconds for {num_images}.")
print("Average inference time per image: {:.6f} seconds".format(avg_inference_time))
print("FPS: {:.2f}".format(fps))

recorder.mark()

In [ ]:
time.sleep(3)
recorder.stop()

In [ ]:
for model in [model1, model2]:
    output_dict_key = f"{model.__class__.__name__}"
    print(f"The NN under test is {output_dict_key}.")

    # Compare outputs using MSE and check if they're the same
    mse = np.mean((cpu_output_dict[output_dict_key] - hls_output_dict[output_dict_key])**2, axis=0)
    is_same = np.allclose(cpu_output_dict[output_dict_key], hls_output_dict[output_dict_key], rtol=1e-5, atol=1e-5)

    print("Comparison of PyTorch vs Original models:")
    print("=" * 50)
    print(f"Are all outputs the same? {'Yes' if is_same else 'No'}")
    print("-" * 50)
    print(f"  Mean Square Error: {mse}")
    print(f"  Average Mean Square Error: {np.mean(mse)}")

In [ ]:
%matplotlib inline
recorder.frame.plot(subplots=True)